In [11]:
import pandas as pd
import pyarrow as pa

In [12]:
import pyarrow as pa

with open(r"D:\Projects\Fashion-Multi-Modal-Rag-System\dataset\polyvore-data-00000-of-00006.arrow", 'rb') as f:
    table = pa.ipc.open_stream(f).read_all()
df = table.to_pandas()
df.head()

,image,category,text,item_ID
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Day Dresses,tibi knit long sleeve dress,100002074_1
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Boots,michael kors leather over-the-knee boots,100002074_2
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Handbags,givenchy leather medium antigona duffel black,100002074_3
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Sunglasses,bottega veneta acetate leather sunglasses,100002074_4
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Floral Decor,pier imports stem,100002074_5


In [13]:
type(df)

pandas.DataFrame

In [14]:
from PIL import Image
from io import BytesIO

# 2. Create a BytesIO object from the bytes
image_stream = BytesIO(df['image'][0]['bytes'])

# 3. Open the image using Pillow
img = Image.open(image_stream)

# 4. Display the image (works on most operating systems)
img.show()


In [15]:
import torch
from transformers import CLIPModel, CLIPProcessor
from tqdm import tqdm

In [16]:
type(pd.DataFrame())

pandas.DataFrame

In [17]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 7490.33it/s]


In [18]:
import os
import io
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dotenv import load_dotenv
import pandas as pd
import pyarrow as pa
from PIL import Image
from PyPDF2 import PdfReader
import torch
from transformers import CLIPModel, CLIPProcessor
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models
from tqdm import tqdm
import numpy as np
from unstructured.partition.pdf import partition_pdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

QDRANT_API_KEY = os.getenv("vdb_api")
QDRANT_ENDPOINT = os.getenv("cluster_endpoint")

TEXT_VECTOR_SIZE = 512
IMAGE_VECTOR_SIZE = 512
TEXT_COLLECTION_NAME = "fashion-text"
IMAGE_COLLECTION_NAME = "fashion-image"
BATCH_SIZE = 32
VERBOSE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_model.to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

text_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased")

qdrant_client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2785.32it/s]


In [19]:
def ensure_collections_exist():
    collections = qdrant_client.get_collections().collections
    collection_names = [c.name for c in collections]

    if TEXT_COLLECTION_NAME not in collection_names:
        if VERBOSE:
            print(f"Creating collection: {TEXT_COLLECTION_NAME}")
        qdrant_client.create_collection(
            collection_name=TEXT_COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=TEXT_VECTOR_SIZE,
                distance=models.Distance.COSINE,
            ),
        )

    if IMAGE_COLLECTION_NAME not in collection_names:
        if VERBOSE:
            print(f"Creating collection: {IMAGE_COLLECTION_NAME}")
        qdrant_client.create_collection(
            collection_name=IMAGE_COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=IMAGE_VECTOR_SIZE,
                distance=models.Distance.COSINE,
            ),
        )


In [26]:
def collections_have_data() -> bool:
    try:
        text_count = qdrant_client.count(collection_name=TEXT_COLLECTION_NAME).count
        image_count = qdrant_client.count(collection_name=IMAGE_COLLECTION_NAME).count
        if VERBOSE:
            print(f"Text collection has {text_count} points")
            print(f"Image collection has {image_count} points")
        return text_count > 0 or image_count > 0
    except Exception:
        return False


def get_text_embedding(text: str) -> np.ndarray:
    return text_model.encode(text)


def get_image_embedding(image: Image.Image) -> np.ndarray:
    inputs = clip_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
    return outputs.pooler_output.cpu().numpy()[0]

In [27]:
def process_pdf_files(pdf_dir: str) -> Tuple[List[Dict], List[Dict]]:
    pdf_path = Path(pdf_dir)
    text_points = []
    image_points = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
    )

    for pdf_file in pdf_path.glob("*.pdf"):
        if VERBOSE:
            print(f"Processing PDF: {pdf_file.name}")

        full_text = ""
        try:
            elements = partition_pdf(
                filename=str(pdf_file),
                extract_images_in_pdf=False,
                infer_table_structure=True,
                strategy="fast",
            )
            texts = [str(el) for el in elements if hasattr(el, 'text') and el.text]
            full_text = " ".join(texts)
        except Exception as e:
            if VERBOSE:
                print(f"  unstructured failed, falling back to PyPDF2: {e}")
            reader = PdfReader(str(pdf_file))
            texts = [page.extract_text() or "" for page in reader.pages]
            full_text = " ".join(texts)

        if full_text.strip():
            chunks = text_splitter.split_text(full_text)
            if VERBOSE:
                print(f"  Split into {len(chunks)} text chunks")
            for i, chunk in enumerate(chunks):
                embedding = get_text_embedding(chunk)
                text_points.append({
                    "id": abs(hash(str(pdf_file.name) + str(i) + chunk[:50])),
                    "vector": embedding.tolist(),
                    "payload": {
                        "source": "pdf",
                        "filename": pdf_file.name,
                        "text": chunk,
                        "chunk_index": i,
                        "category": "fashion_guide",
                    },
                })

    return text_points, image_points



In [28]:
def process_parquet_files(parquet_dir: str) -> Tuple[List[Dict], List[Dict]]:
    data_dir = Path(parquet_dir)
    text_points = []
    image_points = []

    arrow_files = list(data_dir.glob("*.arrow"))
    for arrow_file in arrow_files:
        if VERBOSE:
            print(f"Processing Arrow file: {arrow_file.name}")

        with open(arrow_file, 'rb') as f:
            table = pa.ipc.open_stream(f).read_all()
            df = table.to_pandas()

        for idx, item in tqdm(df.iterrows(), total=len(df), desc=f"Processing {arrow_file.name}"):
            if pd.notna(item.get("text")):
                text = item["text"]
                embedding = get_text_embedding(text)
                text_points.append({
                    "id": abs(hash(str(item.get("item_ID", idx)) + "_text")),
                    "vector": embedding.tolist(),
                    "payload": {
                        "source": "parquet",
                        "item_id": str(item.get("item_ID", "")),
                        "text": text,
                        "category": item.get("category", "unknown"),
                        "type": "text",
                    },
                })

            if item.get("image") and item["image"].get("bytes"):
                try:
                    image = Image.open(io.BytesIO(item["image"]["bytes"]))
                    embedding = get_image_embedding(image)
                    image_points.append({
                        "id": abs(hash(str(item.get("item_ID", idx)) + "_image")),
                        "vector": embedding.tolist(),
                        "payload": {
                            "source": "parquet",
                            "item_id": str(item.get("item_ID", "")),
                            "category": item.get("category", "unknown"),
                            "type": "image",
                        },
                    })
                except Exception as e:
                    print(f"Error processing image at idx {idx}: {e}")
            break
        break

    return text_points, image_points


In [29]:
def batch_upload_text(points: List[Dict], batch_size: int = BATCH_SIZE):
    for i in tqdm(range(0, len(points), batch_size), desc="Uploading text to Qdrant"):
        batch = points[i:i + batch_size]
        qdrant_client.upsert(
            collection_name=TEXT_COLLECTION_NAME,
            points=[
                models.PointStruct(
                    id=p["id"],
                    vector=p["vector"],
                    payload=p["payload"],
                )
                for p in batch
            ],
        )


def batch_upload_image(points: List[Dict], batch_size: int = BATCH_SIZE):
    for i in tqdm(range(0, len(points), batch_size), desc="Uploading images to Qdrant"):
        batch = points[i:i + batch_size]
        qdrant_client.upsert(
            collection_name=IMAGE_COLLECTION_NAME,
            points=[
                models.PointStruct(
                    id=p["id"],
                    vector=p["vector"],
                    payload=p["payload"],
                )
                for p in batch
            ],
        )

In [30]:
def process_and_upload(pdf_dir: str = None, parquet_dir: str = None, force_reingest: bool = False):
    ensure_collections_exist()

    if not force_reingest and collections_have_data():
        print("Data already exists in Qdrant. Skipping ingestion.")
        print("Use force_reingest=True to re-ingest.")
        return

    all_text_points = []
    all_image_points = []

    if pdf_dir and Path(pdf_dir).exists():
        print("Processing PDF files...")
        pdf_text, pdf_images = process_pdf_files(pdf_dir)
        all_text_points.extend(pdf_text)
        all_image_points.extend(pdf_images)
        print(f"Processed {len(pdf_text)} PDF text chunks, {len(pdf_images)} PDF images")

    if parquet_dir and Path(parquet_dir).exists():
        print("Processing Parquet/Arrow files...")
        parquet_text, parquet_images = process_parquet_files(parquet_dir)
        all_text_points.extend(parquet_text)
        all_image_points.extend(parquet_images)
        print(f"Processed {len(parquet_text)} parquet text items, {len(parquet_images)} parquet images")

    if all_text_points:
        print(f"Total text points to upload: {len(all_text_points)}")
        batch_upload_text(all_text_points)
        print("Text upload complete!")

    if all_image_points:
        print(f"Total image points to upload: {len(all_image_points)}")
        batch_upload_image(all_image_points)
        print("Image upload complete!")

    if not all_text_points and not all_image_points:
        print("No data to upload.")

In [32]:

if __name__ == "__main__":
    process_and_upload(
        # pdf_dir=r"D:\Projects\Fashion-Multi-Modal-Rag-System\dataset\dress_pdfs",
        parquet_dir=r"D:\Projects\Fashion-Multi-Modal-Rag-System\dataset",
    )

Creating collection: fashion-text
Creating collection: fashion-image
Text collection has 0 points
Image collection has 0 points
Processing Parquet/Arrow files...
Processing Arrow file: polyvore-data-00000-of-00006.arrow


Processing polyvore-data-00000-of-00006.arrow:   0%|          | 0/18383 [00:00<?, ?it/s]


Processed 1 parquet text items, 1 parquet images
Total text points to upload: 1


Uploading text to Qdrant: 100%|██████████| 1/1 [00:00<00:00,  6.13it/s]


Text upload complete!
Total image points to upload: 1


Uploading images to Qdrant: 100%|██████████| 1/1 [00:00<00:00,  6.27it/s]

Image upload complete!
